## Week 5 Exercise: RAG Implementation
**StudyBuddy:** is a Retrieval-Augmented Generation (RAG) application that allows students to upload textbooks and ask questions about the content. The system extracts text from PDFs, creates embeddings, stores them in a vector database, and uses an LLM to generate contextual answers.

**OCR Requirement**

Tesseract is required for OCR when processing scanned PDFs.
If your PDFs already contain selectable text, OCR may not be necessary.

Install and configure Tesseract as follows:

- macOS

`brew install tesseract`

`export TESSDATA_PREFIX=/opt/homebrew/opt/tesseract/share/tessdata`
- Linux

`sudo apt install tesseract-ocr`

`export TESSDATA_PREFIX=/usr/share/tesseract-ocr/4.00/tessdata`
- Windows (PowerShell)
Install Tesseract from: https://github.com/UB-Mannheim/tesseract/wiki then configure:

`$env:TESSDATA_PREFIX="C:\Program Files\Tesseract-OCR\tessdata"`



You can confirm the installation by running: `tesseract --version` and `tesseract --list-langs` . You should see eng in the list of available languages.


OCR is only used when the PDF contains scanned images instead of text. If Tesseract is not installed, text extraction will still work for digitally generated PDFs.

In [ ]:
#Imports
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from study_buddy import StudyBuddy

In [ ]:
# Check that Tesseract is installed and configured correctly
if "TESSDATA_PREFIX" not in os.environ:
    print("Tesseract language data not configured. OCR may not work.")

In [ ]:
#Setup
load_dotenv(override=True)
MODEL = "gpt-4.1-nano"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
DB_NAME = "studybuddy_db"
RETRIEVAL_K = 3

In [ ]:
SYSTEM_PROMPT = """
You are an AI learning assistant designed to help students understand and quickly find answers within their uploaded textbooks and study materials.
Your primary goal is to help the student learn, understand concepts clearly, and find relevant information from the uploaded documents.
You are provided with context extracted from the student's uploaded textbook or study material. Use ONLY the provided context to answer the student's question.

Guidelines:

1. Answer Using the Provided Context
- Base your answer strictly on the retrieved document context.
- If the answer cannot be found in the context, say:
  "I could not find this information in the uploaded textbook."
- Do NOT invent facts that are not present in the document.

2. Be Clear and Educational
- Explain concepts in a way that a student can easily understand.
- Use simple language and structured explanations.
- If appropriate, break explanations into steps or bullet points.

3. Encourage Understanding
When possible:
- Define key terms
- Explain the reasoning behind the concept
- Provide short examples

4. Reference the Source
If page numbers or sections are available in the metadata, mention them:

Example:
"According to the textbook (Page 42)..."

5. Keep Answers Concise but Helpful
- Provide a direct answer first.
- Follow with a short explanation if needed.
- Avoid unnecessary verbosity.

6. For Conceptual Questions
Explain the concept clearly using the textbook information.

Example format:

Answer:
<short direct answer>

Explanation:
<clear explanation based on the text>

7. For Problem-Solving Questions
If the question involves solving something (math, physics, etc):
- Explain the reasoning step-by-step
- Use formulas or steps if they appear in the textbook

8. If the Question is Outside the Document
Respond politely:

"This question does not appear to be covered in the uploaded textbook."

9. Maintain a Helpful Study Tone
Your tone should feel like a helpful tutor:
- supportive
- clear
- patient
- focused on helping the student learn.

Always prioritize accuracy over guessing.
Always rely on the retrieved document context.

This is the document context you have access to:

{context}
"""

In [ ]:
# Instantiate and launch StudyBuddy UI
studybuddy = StudyBuddy(
    llm=ChatOpenAI(temperature=0, model_name=MODEL),
    embedding_model=EMBEDDING_MODEL,
    db_name=DB_NAME,
    retrieval_k=RETRIEVAL_K,
    system_prompt=SYSTEM_PROMPT
)
studybuddy.launch()